In [ ]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from notebookutils import mssparkutils

In [ ]:
# =========================
# PARAMETERS
# =========================
print("SourceName =", SourceName)
print("IncrementalColumn =", IncrementalColumn)
print("WatermarkDate =", WatermarkDate)
print("LoadType =", LoadType)
print("BusinessKey =", BusinessKey)
bronze_table = f"Bronze_{SourceName}"
silver_table = f"Silver_{SourceName}"
print("Bronze Table =", bronze_table)
print("Silver Table =", silver_table)

In [ ]:
# =========================
# READ BRONZE
# =========================
bronze_df = spark.table(bronze_table)
print("Bronze Count =", bronze_df.count())

In [ ]:
# =========================
# SCHEMA VALIDATION
# =========================
if IncrementalColumn not in bronze_df.columns:
    raise Exception(
f"Incremental Column {IncrementalColumn} not found"
)

In [ ]:
# =========================
# SCHEMA EVOLUTION
# =========================

spark.conf.set(
"spark.databricks.delta.schema.autoMerge.enabled",
"true"
)

In [ ]:
# =========================
# FULL / INCREMENTAL
# =========================

if LoadType == "Full":
    process_df = bronze_df
else:
    process_df = bronze_df.filter(
        col(IncrementalColumn) > WatermarkDate
)

In [ ]:
# =========================
# NO DATA CHECK
# =========================

record_count = process_df.count()

print("Records To Process =", record_count)
if record_count == 0:
    print("No New Records Found")
    mssparkutils.notebook.exit("NO_DATA")

In [ ]:
# =========================
# SILVER TRANSFORMATIONS
# =========================

silver_df = (
process_df
.dropDuplicates()
)

# Optional

silver_df = silver_df.na.drop()

In [ ]:
# =========================
# FIRST LOAD
# =========================

if not spark.catalog.tableExists(silver_table):
    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema","true") \
        .saveAsTable(silver_table)

    print("Silver Table Created")

    mssparkutils.notebook.exit(str(record_count))

In [ ]:
# =========================
# UPSERT / MERGE
# =========================

delta_table = DeltaTable.forName(
spark,
silver_table
)

(
delta_table.alias("t")
.merge(
silver_df.alias("s"),
f"t.{BusinessKey}=s.{BusinessKey}"
)
.whenMatchedUpdateAll()
.whenNotMatchedInsertAll()
.execute()
)

print("Merge Completed")

# =========================
# RETURN RECORD COUNT
# =========================

mssparkutils.notebook.exit(
str(record_count)
)

In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
print(SourceName)

bronze_table = f"Bronze_{SourceName}"
silver_table = f"Silver_{SourceName}"

print("Bronze Table =", bronze_table)
print("Silver Table =", silver_table)

In [ ]:
df = spark.table(bronze_table)

print("Row Count =", df.count())

In [ ]:
df = df.dropDuplicates()

In [ ]:
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(silver_table)

print("Silver Load Completed")

In [1]:
spark.table("Silver_Patients").printSchema()

StatementMeta(, 6266c5b0-a3df-40a3-9c3a-eeb52b23f3e0, 3, Finished, Available, Finished, False)

root
 |-- PatientID: string (nullable = true)
 |-- PatientName: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- DOB: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- InsuranceID: string (nullable = true)
 |-- CreatedDate: string (nullable = true)
 |-- ModifiedDate: string (nullable = true)



In [2]:
spark.table("Silver_Claims").printSchema()

StatementMeta(, 6266c5b0-a3df-40a3-9c3a-eeb52b23f3e0, 4, Finished, Available, Finished, False)

root
 |-- ApprovedAmount: double (nullable = true)
 |-- ClaimAmount: double (nullable = true)
 |-- ClaimDate: long (nullable = true)
 |-- ClaimID: long (nullable = true)
 |-- ClaimStatus: string (nullable = true)
 |-- CreatedDate: long (nullable = true)
 |-- DiagnosisCode: string (nullable = true)
 |-- DoctorID: long (nullable = true)
 |-- HospitalID: long (nullable = true)
 |-- InsuranceProvider: string (nullable = true)
 |-- ModifiedDate: long (nullable = true)
 |-- PatientID: long (nullable = true)



In [3]:
spark.table("Silver_Doctors").printSchema()

StatementMeta(, 6266c5b0-a3df-40a3-9c3a-eeb52b23f3e0, 5, Finished, Available, Finished, False)

root
 |-- DoctorID: string (nullable = true)
 |-- DoctorName: string (nullable = true)
 |-- Specialization: string (nullable = true)
 |-- HospitalID: string (nullable = true)
 |-- ExperienceYears: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- CreatedDate: string (nullable = true)
 |-- ModifiedDate: string (nullable = true)



In [4]:
spark.table("Silver_Hospitals").printSchema()

StatementMeta(, 6266c5b0-a3df-40a3-9c3a-eeb52b23f3e0, 6, Finished, Available, Finished, False)

root
 |-- HospitalID: string (nullable = true)
 |-- HospitalName: string (nullable = true)
 |-- HospitalType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- BedsCount: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- ContactNumber: string (nullable = true)
 |-- CreatedDate: string (nullable = true)
 |-- ModifiedDate: string (nullable = true)



In [5]:
spark.table("Silver_Pharmacy").printSchema()

StatementMeta(, 6266c5b0-a3df-40a3-9c3a-eeb52b23f3e0, 7, Finished, Available, Finished, False)

root
 |-- TransactionID: long (nullable = true)
 |-- PatientID: long (nullable = true)
 |-- MedicineName: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Amount: double (nullable = true)
 |-- PharmacyName: string (nullable = true)
 |-- DoctorID: long (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- CreatedDate: date (nullable = true)
 |-- ModifiedDate: date (nullable = true)

